# Development-calibrated EF point-scale prediction

Select one transferable characteristic time per satellite from Train/SCAN development pixels only, freeze it, and generate predictions for the independent Test pixels. Candidate values are integer days from 1 through 100. Test targets are never used to select the characteristic time.

In [ ]:
from config import configure_runtime
configure_runtime()


In [ ]:
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from config import EF_WORKERS, results_FP
from EF.calibration import calibrate_product, load_calibrated_t, save_calibration
from EF.integrity import atomic_write_csv, clean_matching_files
from EF.prediction import (
    SUMMARY_COLUMNS, canonical_pixel_files, prediction_run_status,
    process_station_file,
)


## Settings and paths

In [ ]:
PRODUCTS = ("ASCAT", "SMAP")
CANDIDATE_DAYS = np.arange(1.0, 101.0)
MINIMUM_PAIRED = 100
SPIN_UP_DAYS = 365.0
STUDY_START = "2015-04-01"
STUDY_END = "2023-12-31"
EXPECTED_DAYS = 3197
TARGET_VARIABLE = "in-situ_RZSM"
GRID_RESOLUTION = 0.1

RESULTS = Path(results_FP)
ISMN_ROOT = RESULTS / "ISMN"
COHORT_ROOT = ISMN_ROOT / "Station_TCA_Screening"
OUTPUT_ROOT = RESULTS / "EF" / "Calibrated_EF"
CALIBRATION_ROOT = OUTPUT_ROOT / "Calibration"
GRID_ROWS = int(round(180.0 / GRID_RESOLUTION))
GRID_COLUMNS = int(round(360.0 / GRID_RESOLUTION))
LATITUDE_AXIS = np.linspace(90.0 - GRID_RESOLUTION / 2.0, -90.0 + GRID_RESOLUTION / 2.0, GRID_ROWS)
LONGITUDE_AXIS = np.linspace(-180.0 + GRID_RESOLUTION / 2.0, 180.0 - GRID_RESOLUTION / 2.0, GRID_COLUMNS)


## Calibrate on development pixels

The same Train cohort used for neural development is evaluated separately for ASCAT and SMAP. Only targets from 2016-03-31 through 2022-04-20 are scored.

In [ ]:
candidate_tables, optimum_tables, calibration_summaries = [], [], []
for product in PRODUCTS:
    candidates, optima, summary = calibrate_product(
        product=product,
        cohort_csv=COHORT_ROOT / "Train_pixel_list.csv",
        point_directory=ISMN_ROOT / "ISMN_Train" / product,
        candidates=CANDIDATE_DAYS,
        minimum_paired=MINIMUM_PAIRED,
    )
    candidate_tables.append(candidates)
    optimum_tables.append(optima)
    calibration_summaries.append(summary)
calibration_files = save_calibration(
    CALIBRATION_ROOT,
    pd.concat(candidate_tables, ignore_index=True),
    pd.concat(optimum_tables, ignore_index=True),
    calibration_summaries,
)
display(pd.DataFrame(calibration_summaries))
print(calibration_files)


## Apply frozen values to independent Test pixels

For the study data, the expected selected values are ASCAT 8 days and SMAP 10 days. The notebook always uses the values loaded from the saved development-only calibration artifact.

In [ ]:
prediction_summaries = []
for product in PRODUCTS:
    selected_t = load_calibrated_t(calibration_files["summary"], product)
    input_directory = ISMN_ROOT / "ISMN_Test" / product
    filenames = canonical_pixel_files(
        COHORT_ROOT / "Test_pixel_list.csv", input_directory,
        STUDY_START, STUDY_END, EXPECTED_DAYS,
    )
    output_directory = OUTPUT_ROOT / "Test" / product
    output_directory.mkdir(parents=True, exist_ok=True)
    clean_matching_files(output_directory, "*_prediction.nc")
    clean_matching_files(output_directory, "*_prediction.nc.tmp.*")
    tasks = [
        {
            "filename": filename,
            "input_directory": input_directory,
            "output_directory": output_directory,
            "latitude_axis": LATITUDE_AXIS,
            "longitude_axis": LONGITUDE_AXIS,
            "data_type": product,
            "target_variable": TARGET_VARIABLE,
            "filter_time_days": selected_t,
            "spin_up_days": SPIN_UP_DAYS,
            "filter_policy": "development_calibrated_sensor_specific_T",
        }
        for filename in filenames
    ]
    workers = min(EF_WORKERS, len(tasks))
    if workers == 1:
        results = [process_station_file(task) for task in tqdm(tasks)]
    else:
        with mp.get_context("spawn").Pool(workers) as pool:
            results = list(tqdm(pool.imap(process_station_file, tasks), total=len(tasks)))
    status = prediction_run_status(filenames, results, output_directory)
    if status["missing_outputs"]:
        raise RuntimeError(f"Missing Calibrated EF outputs: {status['missing_outputs'][:10]}")
    summary = pd.DataFrame(results, columns=SUMMARY_COLUMNS).sort_values(["lat_idx", "lon_idx"])
    summary_file = OUTPUT_ROOT / "Test" / f"Calibrated_EF_{product}_summary.csv"
    atomic_write_csv(summary, summary_file)
    prediction_summaries.append({"product": product, "T_days": selected_t, "pixels": len(summary), "summary": summary_file})
display(pd.DataFrame(prediction_summaries))
